# 03 - Microsoft Entra SDK for Agent Identities

**Learning objectives**
- Understand the Microsoft Entra SDK for Agent ID architecture
- Learn how to deploy the SDK as a containerized service
- Create and manage agent identities from blueprints
- Acquire tokens for autonomous, agent user, and interactive scenarios
- Validate tokens and extract user context
- Call downstream APIs on behalf of agents

**Prerequisites**
- Completed [01-validate-configuration.ipynb](./01-validate-configuration.ipynb)
- Completed [02-token-flows.ipynb](./02-token-flows.ipynb)
- Docker installed (for SDK deployment)
- Understanding of containerization concepts

**What you'll learn:**
- **SDK Architecture**: How the containerized service works
- **Agent Identity CRUD**: Creating individual agent identities from blueprints
- **Token Acquisition Patterns**: Three scenarios (autonomous, agent user, interactive)
- **Token Validation**: Verifying incoming tokens
- **Downstream API Calls**: Using agent tokens to access Microsoft Graph and other APIs

## What is the Microsoft Entra SDK for Agent ID?

The **Microsoft Entra SDK for Agent ID** is a containerized web service that simplifies agent identity management and token operations.

### Why use the SDK?

**Without SDK:**
- Each application must implement OAuth flows
- Token validation logic needs to be duplicated
- Certificate/secret management is complex
- Multi-agent scenarios require custom logic

**With SDK:**
- Simple HTTP API for all identity operations
- Built-in token acquisition and validation
- Automatic credential management
- Support for multiple agents from one blueprint
- Technology-agnostic (works with any language)

### Architecture

```
┌─────────────────┐       HTTP API       ┌──────────────────────┐
│  Your App       │◄────────────────────►│  Agent ID SDK        │
│  (Python/Node)  │  /AuthorizationHeader│  (Container)         │
│                 │  /Validate           │                      │
└─────────────────┘  /Token              └──────────────────────┘
                                                     │
                                                     ▼
                                          ┌──────────────────────┐
                                          │  Azure Entra ID      │
                                          │  + Microsoft Graph   │
                                          └──────────────────────┘
```

### Key Endpoints

1. **`GET /AuthorizationHeader/{serviceName}?AgentIdentity={id}`**
   - Acquires a token for downstream APIs
   - Returns ready-to-use Authorization header
   - Supports autonomous, agent user, and OBO scenarios

2. **`GET /Validate`**
   - Validates incoming user/agent tokens
   - Returns token claims for authorization
   - Used in interactive agent scenarios

3. **`POST /Token`** (alternative endpoint)
   - Returns raw token instead of header
   - Useful for custom token handling

### Deployment Options

- **Docker Container**: Run locally or in any container host
- **Azure Container Instances**: Serverless container hosting
- **Kubernetes**: Production-scale deployments
- **Azure Container Apps**: Managed container service

**Documentation**: [Microsoft Entra SDK for Agent ID](https://learn.microsoft.com/en-us/entra/agent-id/identity-platform/microsoft-entra-sdk-for-agent-identities)

## Setup: Load Configuration

Load the validated configuration from previous notebooks.

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# Load environment
load_dotenv()

config = {
    "tenant_id": os.getenv("AZURE_TENANT_ID"),
    "client_id": os.getenv("AZURE_CLIENT_ID"),
    "credential_type": os.getenv("AZURE_CLIENT_CREDENTIAL_TYPE"),
    "blueprint_id": os.getenv("AGENT_BLUEPRINT_ID"),
    "identifier_uri": os.getenv("AGENT_BLUEPRINT_IDENTIFIER_URI"),
    "scope": os.getenv("AGENT_BLUEPRINT_SCOPE")
}

# Validate
missing = [k for k, v in config.items() if not v]
if missing:
    raise ValueError(f"❌ Missing: {', '.join(missing)}")

print("✅ Configuration loaded")
print(f"   Tenant: {config['tenant_id']}")
print(f"   Client: {config['client_id']}")
print(f"   Blueprint: {config['blueprint_id']}")

## Understanding Agent Identities (Concept)

Unlike the blueprint (which we created with `a365.ps1`), **agent identities** are individual instances that reference the blueprint.

### Why create multiple agent identities?

**Scenario 1: Multi-tenant SaaS**
- One blueprint with shared permissions
- One agent identity per tenant/customer
- Isolated audit trails and monitoring

**Scenario 2: Bot instances**
- One blueprint for "customer service bot"
- One agent identity per user session
- Track which agent handled which conversation

**Scenario 3: Workload isolation**
- One blueprint for "data processing agent"
- One agent identity per workload/job
- Monitor resource usage per identity

### Create Agent Identity API (Preview)

**⚠️ Important**: The agent identity creation endpoint is currently in preview and may not be available in all tenants yet.

**Endpoint**: `POST /v1.0/directory/agentIdentities` *(preview)*

**Required permissions**: `AgentIdentity.ReadWrite.All`

**Request body**:
```json
{
  "displayName": "My Agent Instance",
  "blueprintId": "<blueprint-app-id>",
  "description": "Optional description"
}
```

**For now**, we'll focus on understanding the SDK concepts using your blueprint directly. In production, once the API is available, you would create agent identities as shown below:

In [ ]:
import msal
import requests
from datetime import datetime

# Acquire token for Microsoft Graph
def get_graph_token():
    authority = f"https://login.microsoftonline.com/{config['tenant_id']}"
    
    if config['credential_type'] == 'secret':
        app = msal.ConfidentialClientApplication(
            client_id=config['client_id'],
            client_credential=os.getenv('AZURE_CLIENT_SECRET'),
            authority=authority
        )
    else:
        cert_path = os.getenv('AZURE_CLIENT_CERT_PATH')
        with open(cert_path, 'r') as f:
            private_key = f.read()
        app = msal.ConfidentialClientApplication(
            client_id=config['client_id'],
            client_credential={
                'private_key': private_key,
                'thumbprint': os.getenv('AZURE_CLIENT_CERT_THUMBPRINT')
            },
            authority=authority
        )
    
    result = app.acquire_token_for_client(['https://graph.microsoft.com/.default'])
    if 'access_token' not in result:
        raise Exception(f"Token acquisition failed: {result}")
    return result['access_token']

# Create agent identity (example code)
def create_agent_identity(display_name, description=None):
    """
    Create a new agent identity from the blueprint.
    
    NOTE: This endpoint is currently in preview and may not be available.
    
    Args:
        display_name: Name for this agent identity
        description: Optional description
    
    Returns:
        Agent identity object with 'id' field
    """
    token = get_graph_token()
    url = "https://graph.microsoft.com/v1.0/directory/agentIdentities"
    
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    
    body = {
        "displayName": display_name,
        "blueprintId": config['client_id']  # Blueprint's app ID
    }
    
    if description:
        body["description"] = description
    
    response = requests.post(url, headers=headers, json=body)
    
    if response.status_code == 201:
        return response.json()
    else:
        error = response.json()
        raise Exception(
            f"Failed to create agent identity ({response.status_code}): "
            f"{error.get('error', {}).get('message', error)}"
        )

# Try to create agent identities (will likely fail if endpoint not available)
print("🤖 Attempting to create agent identities...\n")
print("⚠️  Note: This feature is in preview and may not be available in your tenant yet.\n")

agents = []

try:
    # Agent 1: Customer service bot for tenant A
    agent1 = create_agent_identity(
        display_name="Customer Service Bot - Tenant A",
        description="Handles customer inquiries for Tenant A"
    )
    agents.append(agent1)
    print(f"✅ Created Agent 1: {agent1['displayName']}")
    print(f"   ID: {agent1['id']}\n")
    
    # Agent 2: Customer service bot for tenant B
    agent2 = create_agent_identity(
        display_name="Customer Service Bot - Tenant B",
        description="Handles customer inquiries for Tenant B"
    )
    agents.append(agent2)
    print(f"✅ Created Agent 2: {agent2['displayName']}")
    print(f"   ID: {agent2['id']}\n")
    
    # Agent 3: Data processing agent
    agent3 = create_agent_identity(
        display_name="Data Processing Agent",
        description="Processes data exports and reports"
    )
    agents.append(agent3)
    print(f"✅ Created Agent 3: {agent3['displayName']}")
    print(f"   ID: {agent3['id']}\n")
    
    print(f"✅ Successfully created {len(agents)} agent identities")
    
    # Save agent IDs for later use
    agent_ids = [agent['id'] for agent in agents]
    with open('agent-identities.json', 'w') as f:
        json.dump(agents, f, indent=2)
    print(f"💾 Agent details saved to: agent-identities.json")
    
except Exception as e:
    print(f"❌ Agent identity creation not available: {e}\n")
    print("📚 This is expected - the agent identity endpoint is in preview.")
    print("   We'll use your actual blueprint directly to demonstrate SDK concepts.\n")
    
    # Use the actual blueprint as the agent identity
    print("🔗 Using your Agent Blueprint as the agent identity...\n")
    
    # Get the actual service principal for your blueprint
    principal_id = os.getenv('AGENT_BLUEPRINT_PRINCIPAL_ID')
    
    agents = [
        {
            "id": principal_id,  # Real service principal ID from a365.ps1
            "displayName": "My Agent Identity Blueprint",
            "description": "Your actual Agent Blueprint from a365.ps1",
            "blueprintId": config['client_id'],
            "appId": config['client_id'],
            "objectId": config['blueprint_id'],
            "isBlueprint": True  # This is the blueprint itself
        }
    ]
    
    print(f"   ✅ Blueprint Agent Identity:")
    print(f"      Display Name: {agents[0]['displayName']}")
    print(f"      Service Principal ID: {principal_id}")
    print(f"      Application ID: {config['client_id']}")
    print(f"      Blueprint ID: {config['blueprint_id']}")
    
    print(f"\n✅ Using real Azure resources for SDK demonstrations")
    print("   All token acquisitions and API calls will use your actual blueprint")

## Using the Microsoft Entra SDK for Agent ID

Now let's explore how applications interact with the SDK to acquire tokens for agents.

### SDK Configuration

The SDK requires configuration that matches your `.env` setup:

```json
{
  "AzureAd": {
    "Instance": "https://login.microsoftonline.com/",
    "TenantId": "<your-tenant-id>",
    "ClientId": "<blueprint-client-id>",
    "ClientCredentials": [
      {
        "SourceType": "ClientSecret",
        "ClientSecret": "<your-secret>"
      }
    ]
  },
  "DownstreamApis": {
    "Graph": {
      "BaseUrl": "https://graph.microsoft.com/v1.0",
      "Scopes": "https://graph.microsoft.com/.default"
    }
  }
}
```

### Three Token Acquisition Patterns

The SDK supports three scenarios based on the documentation:

### Pattern 1: Autonomous Agent (App-Only)

The agent acts on its own behalf.

**API Call**:
```http
GET /AuthorizationHeader/Graph?AgentIdentity=<agent-id>
Authorization: Bearer <blueprint-token>
```

**Parameters**:
- `AgentIdentity`: The specific agent identity ID
- Authorization header: Token for the blueprint application

**Response**:
```json
{
  "authorizationHeader": "Bearer eyJ0eXAiOiJKV1Qi..."
}
```

Let's simulate this pattern:

In [ ]:
# Simulate SDK's autonomous agent token acquisition

def simulate_sdk_autonomous_token(agent_identity_id):
    """
    Simulates how the SDK acquires tokens for autonomous agents.
    
    In production, this would be:
    GET http://sdk-host:port/AuthorizationHeader/Graph?AgentIdentity={agent_identity_id}
    """
    print(f"📡 [SDK] Acquiring token for agent: {agent_identity_id}")
    
    # SDK would internally do this:
    # 1. Validate the agent identity exists and belongs to this blueprint
    # 2. Use the blueprint's credentials to acquire a token
    # 3. Include agent context in the token request
    # 4. Return the authorization header
    
    # Acquire REAL token using blueprint credentials
    token = get_graph_token()  # This uses your actual client secret/cert from .env
    
    auth_header = f"Bearer {token}"
    
    print(f"   ✅ Token acquired for agent")
    print(f"   Token preview: {token[:40]}...{token[-20:]}")
    print(f"   Authorization Header: Bearer <token>\n")
    
    return auth_header

# Example: Use your blueprint as an agent identity
print("🧪 Pattern 1: Autonomous Agent Tokens\n")
print(f"Using your Agent Blueprint: {config['client_id']}\n")

# Use your actual blueprint
agent = agents[0]
auth_header = simulate_sdk_autonomous_token(agent['id'])

# Now make a REAL API call to demonstrate it works
print(f"   🔍 Making real Microsoft Graph API call...")
try:
    import requests
    url = f"https://graph.microsoft.com/v1.0/servicePrincipals/{agent['id']}"
    response = requests.get(url, headers={"Authorization": auth_header})
    
    if response.status_code == 200:
        sp = response.json()
        print(f"   ✅ Successfully queried service principal")
        print(f"      Display Name: {sp.get('displayName')}")
        print(f"      App ID: {sp.get('appId')}")
        print(f"      Service Principal Type: {sp.get('servicePrincipalType')}\n")
    else:
        print(f"   ℹ️  API call returned: {response.status_code}\n")
except Exception as e:
    print(f"   ℹ️  API call demo: {e}\n")

print(f"   ✅ Your blueprint is functioning as an agent identity")
print(f"   📝 In production with multiple agent identities, each would get its own token")

### Pattern 2: Autonomous Agent User

The agent acts as a dedicated user (e.g., agent with mailbox).

**API Call**:
```http
GET /AuthorizationHeader/Graph?AgentIdentity=<agent-id>&AgentUserId=<user-object-id>
Authorization: Bearer <blueprint-token>
```

**OR**:
```http
GET /AuthorizationHeader/Graph?AgentIdentity=<agent-id>&AgentUsername=<upn>
Authorization: Bearer <blueprint-token>
```

**Parameters**:
- `AgentIdentity`: The agent identity ID (required)
- `AgentUserId`: User object ID (provide either this OR AgentUsername)
- `AgentUsername`: User Principal Name (provide either this OR AgentUserId)

**Use case**: Agent needs to send emails, create calendar events, etc. as itself

**Note**: You cannot provide both `AgentUserId` and `AgentUsername` - validation error will occur.

In [ ]:
# Simulate SDK's agent user token acquisition

def simulate_sdk_agent_user_token(agent_identity_id, user_id=None, username=None):
    """
    Simulates how the SDK acquires tokens for agent users.
    """
    if not user_id and not username:
        raise ValueError("Must provide either user_id or username")
    if user_id and username:
        raise ValueError("Cannot provide both user_id and username")
    
    identifier = f"UserId={user_id}" if user_id else f"Username={username}"
    print(f"📡 [SDK] Acquiring token for agent user: {identifier}")
    
    # SDK would:
    # 1. Validate agent identity
    # 2. Resolve user by ID or UPN
    # 3. Acquire token with user context
    # 4. Return authorization header
    
    token = get_graph_token()
    auth_header = f"Bearer {token}"
    
    print(f"   ✅ Token acquired with user context")
    print(f"   Agent can now act as this user\n")
    
    return auth_header

# Example: Agent user scenario
print("🧪 Pattern 2: Autonomous Agent User\n")

# Simulate an agent that has its own user account
# This would be a real user object ID from your tenant
agent_user_id = "example-user-object-id"
agent_username = "agent.bot@contoso.com"

print("📧 Scenario: Agent needs to send email as itself\n")

# Option 1: Using user object ID
print("   Option 1: Using AgentUserId parameter")
print(f"   API Call: GET /AuthorizationHeader/Graph?AgentIdentity={agents[0]['id']}&AgentUserId={{user_id}}\n")

# Option 2: Using username (UPN)
print("   Option 2: Using AgentUsername parameter")
print(f"   API Call: GET /AuthorizationHeader/Graph?AgentIdentity={agents[0]['id']}&AgentUsername={{upn}}\n")

print("   ⚠️  Important: Cannot provide both AgentUserId and AgentUsername\n")
print("   📝 This pattern requires the agent to have an associated user account")

### Pattern 3: Interactive Agent (On-Behalf-Of)

The agent acts on behalf of a human user.

**Flow**:
1. User authenticates to your app
2. App receives user token
3. App sends request to agent API with user token
4. Agent validates user token via SDK `/Validate` endpoint
5. Agent exchanges user token for downstream API token via `/AuthorizationHeader`

**Step 1: Validate user token**:
```http
GET /Validate
Authorization: Bearer <user-token>
```

**Response**:
```json
{
  "claims": {
    "aud": "api://blueprint-id",
    "sub": "user-object-id",
    "name": "John Doe",
    "...": "..."
  }
}
```

**Step 2: Get authorization header on behalf of user**:
```http
GET /AuthorizationHeader/Graph?AgentIdentity=<agent-id>
Authorization: Bearer <user-token>
```

**Response**:
```json
{
  "authorizationHeader": "Bearer <downstream-token>"
}
```

In [ ]:
# Simulate SDK's interactive agent (OBO) flow

def simulate_sdk_validate_user_token(user_token):
    """
    Simulates the SDK's /Validate endpoint.
    """
    print(f"📡 [SDK] Validating user token...")
    
    # SDK would:
    # 1. Validate JWT signature
    # 2. Check audience matches blueprint
    # 3. Verify issuer is trusted
    # 4. Check expiration
    # 5. Return claims
    
    # For demo, decode without validation
    import jwt
    try:
        claims = jwt.decode(user_token, options={"verify_signature": False})
        print(f"   ✅ Token validated")
        print(f"   User: {claims.get('name', claims.get('preferred_username', 'Unknown'))}")
        print(f"   Object ID: {claims.get('oid', 'N/A')}\n")
        return claims
    except Exception as e:
        print(f"   ❌ Validation failed: {e}\n")
        raise

def simulate_sdk_obo_token(agent_identity_id, user_token):
    """
    Simulates the SDK's OBO token acquisition.
    """
    print(f"📡 [SDK] Acquiring token on behalf of user...")
    
    # SDK would:
    # 1. Validate incoming user token
    # 2. Extract user claims
    # 3. Use OBO flow to exchange for downstream token
    # 4. Return authorization header
    
    # For demo, just show the concept
    print(f"   🔄 Exchanging user token for downstream API token")
    print(f"   📋 Agent Identity: {agent_identity_id}")
    print(f"   ✅ OBO token acquired\n")
    
    return "Bearer <obo-token>"

# Example: Interactive agent scenario
print("🧪 Pattern 3: Interactive Agent (On-Behalf-Of)\n")
print("📧 Scenario: User asks agent to send email on their behalf\n")

# Step 1: User authenticates and app gets user token
print("Step 1: User signs in to client app")
print("   Client app receives user token with audience = agent blueprint\n")

# Step 2: Client app calls agent API with user token
print("Step 2: Client app sends request to agent API")
print("   POST /api/sendEmail")
print("   Authorization: Bearer <user-token>\n")

# Step 3: Agent validates user token
print("Step 3: Agent validates user token via SDK")
print("   GET /Validate")
print("   Authorization: Bearer <user-token>\n")

# Simulate with a sample token (in reality, this would be from user auth)
sample_user_token = get_graph_token()  # Just for demo
# user_claims = simulate_sdk_validate_user_token(sample_user_token)

# Step 4: Agent acquires downstream token
print("Step 4: Agent acquires token for Microsoft Graph (OBO)")
print(f"   GET /AuthorizationHeader/Graph?AgentIdentity={agents[0]['id']}")
print("   Authorization: Bearer <user-token>\n")

# downstream_auth = simulate_sdk_obo_token(agents[0]['id'], sample_user_token)
print("   ✅ Agent now has token to call Graph on behalf of user\n")

# Step 5: Agent calls Graph to send email
print("Step 5: Agent calls Microsoft Graph to send email")
print("   POST /v1.0/me/sendMail")
print("   Authorization: Bearer <obo-token>")
print("   (Email sent from user's mailbox, not agent's)\n")

print("✅ Complete OBO flow: User → Client → Agent → SDK → Graph")

## Monitoring and Logging

One key benefit of using agent identities is the ability to monitor and audit agent actions separately.

### What you can monitor:

1. **Sign-in logs**: Track agent authentication events
2. **Audit logs**: See what actions each agent performed
3. **Token metrics**: Monitor token issuance and validation
4. **API usage**: Track which APIs each agent called
5. **Error rates**: Identify problematic agents

### Where to find logs:

- **Azure Portal**: Entra ID → Monitoring → Sign-in logs
- **Microsoft Graph API**: `/auditLogs/signIns` and `/auditLogs/directoryAudits`
- **Azure Monitor**: Configure diagnostic settings for advanced analytics

### Query agent-specific logs via Graph:

In [ ]:
# Query sign-in logs for our agents

def query_agent_signin_logs(agent_app_id, days=7):
    """
    Query Microsoft Graph for sign-in logs filtered by agent app ID.
    
    Args:
        agent_app_id: Application ID to filter by
        days: Number of days to look back
    
    Returns:
        List of sign-in events
    """
    from datetime import datetime, timedelta
    
    token = get_graph_token()
    
    # Calculate date filter
    start_date = (datetime.utcnow() - timedelta(days=days)).strftime('%Y-%m-%dT%H:%M:%SZ')
    
    # Query with filter
    url = "https://graph.microsoft.com/v1.0/auditLogs/signIns"
    params = {
        "$filter": f"appId eq '{agent_app_id}' and createdDateTime ge {start_date}",
        "$top": 10,
        "$orderby": "createdDateTime desc"
    }
    
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    
    response = requests.get(url, headers=headers, params=params)
    
    if response.status_code == 200:
        return response.json().get('value', [])
    else:
        error = response.json()
        raise Exception(
            f"Failed to query sign-in logs ({response.status_code}): "
            f"{error.get('error', {}).get('message', error)}"
        )

# Query recent sign-ins for our blueprint
print("📊 Querying agent sign-in logs...\n")

try:
    signins = query_agent_signin_logs(
        agent_app_id=config['client_id'],
        days=7
    )
    
    if signins:
        print(f"✅ Found {len(signins)} sign-in event(s):\n")
        for signin in signins[:5]:  # Show first 5
            print(f"   📅 {signin.get('createdDateTime')}")
            print(f"   📱 App: {signin.get('appDisplayName')}")
            print(f"   🔐 Status: {signin.get('status', {}).get('errorCode', 0)}")
            print(f"   🌐 IP: {signin.get('ipAddress', 'N/A')}")
            print()
    else:
        print("ℹ️  No recent sign-in events found")
        print("   This is normal for newly created blueprints\n")
        
except Exception as e:
    print(f"❌ Failed to query logs: {e}")
    print("\n⚠️  Note: Requires AuditLog.Read.All permission")
    print("   You may need to grant this permission in Azure Portal")

## Cleanup: Delete Agent Identities

When you're done testing, clean up the agent identities we created.

**Note**: The blueprint itself (from `a365.ps1`) remains - only the identities are deleted.

In [ ]:
# Delete agent identities (only if real ones were created)

def delete_agent_identity(agent_id):
    """
    Delete an agent identity.
    
    Args:
        agent_id: The agent identity ID to delete
    """
    token = get_graph_token()
    url = f"https://graph.microsoft.com/v1.0/directory/agentIdentities/{agent_id}"
    
    headers = {
        "Authorization": f"Bearer {token}"
    }
    
    response = requests.delete(url, headers=headers)
    
    if response.status_code == 204:
        return True
    elif response.status_code == 404:
        print(f"   ⚠️  Agent already deleted or not found")
        return False
    else:
        error = response.json()
        raise Exception(
            f"Failed to delete agent ({response.status_code}): "
            f"{error.get('error', {}).get('message', error)}"
        )

# Only try to delete if we have real agents (not the blueprint itself)
if agents and not agents[0].get('isBlueprint'):
    print("🗑️  Cleaning up agent identities...\n")
    
    for agent in agents:
        try:
            print(f"   Deleting: {agent['displayName']}")
            delete_agent_identity(agent['id'])
            print(f"   ✅ Deleted\n")
        except Exception as e:
            print(f"   ❌ Failed: {e}\n")
    
    print("✅ Cleanup complete")
    
    # Remove saved file
    if Path('agent-identities.json').exists():
        Path('agent-identities.json').unlink()
        print("🗑️  Removed agent-identities.json")
else:
    print("ℹ️  No agent identities to clean up")
    print("   We used your blueprint directly - it remains active")
    print("   To delete the blueprint itself, use Azure Portal or the a365.ps1 script")

## Summary

✅ **What you've learned:**

1. **SDK Architecture**: Containerized service that simplifies agent identity management
2. **Agent Identities**: Creating multiple agent instances from a single blueprint
3. **Token Patterns**: Three scenarios (autonomous, agent user, interactive/OBO)
4. **Token Validation**: Validating incoming user tokens
5. **Monitoring**: Querying sign-in logs and audit trails
6. **Cleanup**: Managing agent identity lifecycle

**Key concepts:**
- The SDK exposes simple HTTP APIs for complex OAuth flows
- Multiple agents can share one blueprint but have separate identities
- Different token acquisition patterns for different use cases
- Agent actions are auditable and monitorable separately

**Production deployment:**
- Deploy SDK as Docker container
- Configure with your blueprint credentials
- Applications call SDK HTTP endpoints instead of implementing OAuth
- SDK handles token caching, refresh, and validation

**Next steps:**
- **[04-interactive-authentication.ipynb](./04-interactive-authentication.ipynb)**: Deep dive into interactive agent authentication and user authorization flows

## How Identity Propagates from the Blueprint

According to [Microsoft's official documentation](https://learn.microsoft.com/en-us/entra/agent-id/identity-platform/agent-blueprint), agent identity blueprints serve four key purposes:

### 1. Blueprint as Template
All agent identities in a Microsoft Entra ID tenant are created from an agent identity blueprint. The blueprint records common characteristics so that all agent identities have a consistent configuration, including:
- `description`: Brief summary of the agent's purpose and functions
- `appRoles`: Define roles that can be given to users and principals
- `verifiedPublisher`: The organization that built the agent
- Authentication protocol settings like `optionalClaims`

### 2. Blueprint Creates Agent Identities
The blueprint is not just a configuration template—it's a special identity type that can provision and deprovision agent identities:
- Has an OAuth client ID for requesting access tokens
- Has credentials (secret/certificate) for authentication
- Requires `AgentIdentity.CreateAsManager` permission to create agent identities via Microsoft Graph APIs

### 3. Blueprint Holds Credentials for Agent Identities
**Key insight**: Each agent identity does NOT have its own credentials. Instead:
- Credentials are configured on the blueprint
- When an AI agent needs to perform an operation, the blueprint's credentials are used to request an access token from Microsoft Entra ID
- All agent identities created from the same blueprint share the same credentials

### 4. Blueprint as Container for Management
The blueprint provides a logical container where administrators can apply policies and settings that affect all agent identities:
- **OAuth permissions** granted to a blueprint are granted to all its agent identities
- **Conditional access policies** applied to a blueprint affect all its agent identities
- **Disabling** a blueprint prevents all its agent identities from authenticating

**In summary**: One place to manage credentials and permissions (the blueprint), many agent identities that leverage them.


In [ ]:
# Inspect blueprint permissions and token roles

import os
import requests
import jwt

# Acquire app-only token using the blueprint credentials
bp_token = get_graph_token()

bp_app_id = config['client_id']
bp_principal_id = os.getenv('AGENT_BLUEPRINT_PRINCIPAL_ID')

print("🔎 Inspecting blueprint configuration and effective permissions\n")

headers = {
    "Authorization": f"Bearer {bp_token}",
    "Content-Type": "application/json"
}

# 1) Get the App Registration (application) by appId to inspect declared requiredResourceAccess
app_url = f"https://graph.microsoft.com/v1.0/applications"
app_resp = requests.get(app_url, headers=headers, params={"$filter": f"appId eq '{bp_app_id}'"})
if app_resp.status_code == 200 and app_resp.json().get('value'):
    app_obj = app_resp.json()['value'][0]
    print(f"✅ Found Application: {app_obj.get('displayName')} ({bp_app_id})")
    rra = app_obj.get('requiredResourceAccess', [])
    
    # Resolve Microsoft Graph app roles to names for readability
    graph_sp_resp = requests.get(
        "https://graph.microsoft.com/v1.0/servicePrincipals",
        headers=headers,
        params={"$filter": "appId eq '00000003-0000-0000-c000-000000000000'"}
    )
    graph_roles_by_id = {}
    if graph_sp_resp.status_code == 200 and graph_sp_resp.json().get('value'):
        graph_sp = graph_sp_resp.json()['value'][0]
        for role in graph_sp.get('appRoles', []):
            graph_roles_by_id[role.get('id')] = role.get('value') or role.get('displayName')
    
    print("\n📋 Declared required resource access (by Application):")
    for entry in rra:
        resource_app_id = entry.get('resourceAppId')
        accesses = entry.get('resourceAccess', [])
        if resource_app_id == '00000003-0000-0000-c000-000000000000':
            print("  - Microsoft Graph:")
            for acc in accesses:
                role_name = graph_roles_by_id.get(acc.get('id'), acc.get('id'))
                print(f"      • {role_name} (type={acc.get('type')})")
        else:
            print(f"  - Resource {resource_app_id}:")
            for acc in accesses:
                print(f"      • {acc.get('id')} (type={acc.get('type')})")
else:
    print(f"ℹ️ Could not retrieve Application by appId. Status: {app_resp.status_code}")

# 2) Get Service Principal assignments to see actual admin-consented app roles
if bp_principal_id:
    spa_url = f"https://graph.microsoft.com/v1.0/servicePrincipals/{bp_principal_id}/appRoleAssignments"
    spa_resp = requests.get(spa_url, headers=headers)
    if spa_resp.status_code == 200:
        assignments = spa_resp.json().get('value', [])
        if assignments:
            print("\n🔐 Admin-consented app role assignments (by Service Principal):")
            for a in assignments:
                resource_id = a.get('resourceId')
                role_id = a.get('appRoleId')
                # Try resolve resource display name
                res_resp = requests.get(
                    f"https://graph.microsoft.com/v1.0/servicePrincipals/{resource_id}",
                    headers=headers
                )
                resource_name = None
                role_name = None
                if res_resp.status_code == 200:
                    res_sp = res_resp.json()
                    resource_name = res_sp.get('displayName')
                    # Map role id to a name if it's Microsoft Graph
                    if res_sp.get('appId') == '00000003-0000-0000-c000-000000000000':
                        for role in res_sp.get('appRoles', []):
                            if role.get('id') == role_id:
                                role_name = role.get('value') or role.get('displayName')
                print(f"  • {resource_name or resource_id} → {role_name or role_id}")
        else:
            print("\nℹ️ No app role assignments found yet (admin consent may be pending).")
    else:
        print(f"\nℹ️ Unable to query SP assignments. Status: {spa_resp.status_code}")
else:
    print("\nℹ️ AGENT_BLUEPRINT_PRINCIPAL_ID not set in environment.")

# 3) Decode the app-only token to show effective roles used for Graph
try:
    claims = jwt.decode(bp_token, options={"verify_signature": False})
    roles = claims.get('roles', [])
    aud = claims.get('aud')
    appid = claims.get('appid')
    print("\n🧾 Token claims (app-only access token for Graph):")
    print(f"  aud: {aud}")
    print(f"  appid (blueprint app): {appid}")
    if roles:
        print("  roles:")
        for r in roles:
            print(f"    • {r}")
    else:
        print("  roles: (none) — ensure application permissions are granted and consented")
except Exception as e:
    print(f"\nℹ️ Unable to decode token: {e}")

print("\n✅ Propagation confirmed: tokens and assignments come from the blueprint; agent identities inherit these permissions.")


In [ ]:
# Agent → Blueprint linking demo (context propagation)

import os

def link_agent_to_blueprint(agent_identity_id: str, blueprint_app_id: str):
    """
    Conceptual link object that an SDK (or your service) would track.
    In production, real agent identities include this linkage automatically.
    """
    return {
        "agentIdentityId": agent_identity_id,
        "blueprintAppId": blueprint_app_id,
        "blueprintPrincipalId": os.getenv('AGENT_BLUEPRINT_PRINCIPAL_ID'),
        "credentialSource": "blueprint",
        "permissions": "inherited-from-blueprint"
    }


def get_authorization_for_agent(agent_link: dict):
    """Return headers used to call downstream APIs, including optional context headers."""
    auth = f"Bearer {get_graph_token()}"
    # Microsoft Graph ignores custom headers; include them for YOUR services to log per-agent context
    return {
        "Authorization": auth,
        "X-Agent-Identity": agent_link["agentIdentityId"],
        "X-Blueprint-AppId": agent_link["blueprintAppId"]
    }

# Demo: use the blueprint principal id as a stand-in for an agent identity id
agent_identity_id = os.getenv('AGENT_BLUEPRINT_PRINCIPAL_ID') or "example-agent-id"
agent_link = link_agent_to_blueprint(agent_identity_id, config['client_id'])
headers = get_authorization_for_agent(agent_link)

print("🔗 Agent → Blueprint link:")
print(f"  agentIdentityId: {agent_link['agentIdentityId']}")
print(f"  blueprintAppId:  {agent_link['blueprintAppId']}")
print(f"  principalId:     {agent_link['blueprintPrincipalId']}\n")

print("📡 Example calls:")
print("  • To Microsoft Graph: use only 'Authorization' header")
print("    GET https://graph.microsoft.com/v1.0/me")
print("    Authorization: Bearer <token>\n")

print("  • To YOUR APIs: include agent context headers for auditing")
print("    POST https://your-api/agents/action")
print("    Authorization: Bearer <token>")
print("    X-Agent-Identity: <agent-identity-id>")
print("    X-Blueprint-AppId: <blueprint-app-id>\n")

print("✅ This shows how per-agent context travels alongside the shared blueprint authorization.")


## Agent Identity Propagation: Blueprint to Agent Identities

To visualize how agent identities inherit permissions and credentials from the blueprint, here's the official diagram from Microsoft Learn:

![Agent Identity Blueprint](https://learn.microsoft.com/en-us/entra/agent-id/identity-platform/media/agent-blueprint/agent-blueprint.png)

### Key Concepts from the Documentation:

**Blueprint as Template**
- All agent identities are created from an agent identity blueprint
- The blueprint records common characteristics shared by all agent identities
- Properties like `description`, `appRoles`, `verifiedPublisher`, and `optionalClaims` are defined once at the blueprint level

**Shared Credentials**
- Each agent identity doesn't have its own credentials
- Instead, credentials are configured on the blueprint and shared by all agent identities
- When an agent needs to perform an operation, the blueprint's credentials are used to request access tokens

**Permissions Inheritance**
- OAuth permissions granted to a blueprint are automatically granted to all its agent identities
- Conditional access policies applied to a blueprint affect all its agent identities
- This provides a single point of administration for managing permissions at scale

**Blueprint as Container**
- The blueprint provides a logical container for agent identities
- Administrators can apply policies and settings to the blueprint that cascade to all agent identities
- Disabling a blueprint prevents all its agent identities from authenticating


In [ ]:
# Render Official Propagation Diagram

from IPython.display import Image, display

# Display the official Microsoft Learn diagram
diagram_url = "https://learn.microsoft.com/en-us/entra/agent-id/identity-platform/media/agent-blueprint/agent-blueprint.png"

print("🖼️ Displaying official agent identity blueprint diagram from Microsoft Learn\n")
display(Image(url=diagram_url, width=800))

print("\n📚 Documentation: https://learn.microsoft.com/en-us/entra/agent-id/identity-platform/agent-blueprint")
